# Marimba Mallet Fine Tuning



[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alicater/mallet_fine_tuning/blob/main/Mallet_Fine_Tuning.ipynb) 

# Overview
This project's goal is to create a fullstack web app that interfaces with a finetuned small languagle model (the Llama 3.2-3B) trained on a large amount of marimba mallet descriptions and the Leigh Howard Stevens book *Method of Movement* but also hooked up to a RAG database. The model should be able to (at least somewhat) accurately answer questions surrounding marimba methodology and mallet differences.

We used playwright to scrape the Steve Weiss website (an online percussion equipment store) for marimba mallet names, model number, and description to create a large marimba dataset. After collecting the data, we use Google Gemini's model to create queries from the mallet database and the method of movement book.We also use Chromadb for a vectordatabase for our RAG implementation.

Our websraping, querie creation, and fine-tuning is all done within a Google colab notebook while our backend is written in Python and frontend in React.

# Webscraping

Gathers mallet names, model number (when applicable), description, and link from Steveweiss-an online percussion equipment store.

In [ ]:
# installing libraries needed for webscraping
!pip install playwright
!playwright install chromium
!playwright install-deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 46.3 MB/s eta 0:00:00
(node:8198) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)
167.3 MiB [] 0% 340.0s167.3 MiB [] 0% 22.3s167.3 MiB [] 0% 8.8s167.3 MiB [] 1% 3.7s167.3 MiB [] 2% 2.8s167.3 MiB [] 3% 2.2s167.3 MiB [] 4% 2.0s167.3 MiB [] 6% 1.8s167.3 MiB [] 7% 1.6s167.3 MiB [] 8% 1.6s167.3 MiB [] 9% 1.5s167.3 MiB [] 11% 1.4s167.3 MiB [] 12% 1.3s167.3 MiB [] 14% 1.3s167.3 MiB [] 15% 1.2s167.3 MiB [] 16% 1.2s167.3 MiB [] 18% 1.1s167.3 MiB [] 19% 1.1s167.3 MiB [] 21% 1.1s167.3 MiB [] 22% 1.1s167.3 MiB [] 23% 1.0s167.3 MiB [] 24% 1.0s167.3 MiB [] 25% 1.0s167.3 MiB [] 27% 1.0s167.3 MiB [] 28% 1.0s167.3 MiB [] 29% 1.0s167.3 MiB [] 31% 0.9s167.3 MiB [] 32% 0.9s167.3 MiB [] 34% 0.9s167.3 MiB [] 35% 0.9

In [ ]:
import asyncio
from playwright.async_api import async_playwright
import pandas as pd
import nest_asyncio

async def scrape_mallets():
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=["--no-sandbox", "--disable-setuid-sandbox"]
        )
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"
        )
        page = await context.new_page()

        all_mallets = []
        links = []
        base_url = "https://www.steveweissmusic.com/collections/marimba-mallets"

        # --- Collect All Mallet links first ---
        print("Starting link collection across all pages...")
        for page_num in range(1, 20):  # gets up to 20 pages
            page_url = f"{base_url}?page={page_num}"
            print(f"  Checking Page {page_num}...")

            try:
                await page.goto(page_url, wait_until="networkidle", timeout=60000)
                # fake scrolls to ensure loading of all products
                await page.mouse.wheel(0, 4000)
                await asyncio.sleep(2)

                elements = await page.query_selector_all("a[href*='/products/']")
                new_links_on_page = 0

                for el in elements:
                    href = await el.get_attribute("href")
                    if href and "/products/" in href and "collections" not in href:
                        # Clean URL (remove fragment/query params)
                        clean_href = href.split('?')[0].split('#')[0]
                        full_url = "https://www.steveweissmusic.com" + clean_href if clean_href.startswith("/") else clean_href

                        if full_url not in links:
                            links.append(full_url)
                            new_links_on_page += 1

                print(f"    Added {new_links_on_page} new links. Total so far: {len(links)}")

                if new_links_on_page == 0:
                    print("    No new links found. Ending pagination.")
                    break
            except Exception as e:
                print(f"    Error on page {page_num}: {e}")
                break

        print(f"\nPhase 1 Complete: Found {len(links)} unique mallet links.")

        # --- Deep Scrape all ---
        print("\nStarting deep scrape of product details...")

        for i, link in enumerate(links):
            # refreshing page every 100 links to prevent colab memory leaks
            if i > 0 and i % 100 == 0:
                await page.close()
                page = await context.new_page()

            try:
                await page.goto(link, wait_until="domcontentloaded", timeout=30000)

                # Extract Name
                name_handle = await page.query_selector("h1")
                name = (await name_handle.inner_text()).strip() if name_handle else "Unknown Mallet"

                # Extract Model Info
                model_no = "N/A"
                model_locator = page.get_by_text("Model:").first
                if await model_locator.count() > 0:
                    model_raw = await model_locator.inner_text()
                    model_no = model_raw.replace("Model:", "").strip()

                # Extract Description
                description = "No description found."
                selectors = ["#product-description", ".product-description", ".description-content"]

                for selector in selectors:
                    try:
                        element = await page.wait_for_selector(selector, timeout=2500)
                        if element:
                            await asyncio.sleep(0.5)
                            raw_text = await element.inner_text()
                            if len(raw_text.strip()) > 25:
                                description = raw_text.strip()
                                break
                    except:
                        continue

                if description == "No description found.":
                    main_body = await page.query_selector("main")
                    if main_body:
                        full_text = await main_body.inner_text()
                        if "Description" in full_text:
                            description = full_text.split("Description")[-1].split("Product Info")[0].strip()

                description = description.replace("Description", "", 1).strip()

                # save to list
                all_mallets.append({
                    "Name": name,
                    "Model": model_no,
                    "Description": description,
                    "URL": link
                })

                print(f"[{i+1}/{len(links)}] ✅ {name} | Desc: {len(description)} chars")

                # autosaves every 25 mallets just in case
                if (i + 1) % 25 == 0:
                    pd.DataFrame(all_mallets).to_csv("marimba_mallets_dataset.csv", index=False)
                    print("      Progress autosaved...")

                await asyncio.sleep(0.8) #add jitter delay

            except Exception as e:
                print(f"  ❌ Error on {link}: {e}")

        await browser.close()

        # Final Save
        if all_mallets:
            df = pd.DataFrame(all_mallets)
            df.to_csv("marimba_mallets_dataset.csv", index=False)
            print(f"\nFinal Success! Total mallets saved: {len(df)}")
        else:
            print("\nScrape complete, but no data was collected.")

nest_asyncio.apply()
asyncio.run(scrape_mallets())

# Generating the queries

Uses Google Gemini API to format and create queries for fine-tuning using the mallet data collected from Steveweiss.

In [ ]:
from google.colab import userdata
import google.generativeai as genai

# This pulls the key from the secrets sidebar safely -- I turned this key off, please let me know if we need to run this again
api_key = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=api_key)

model = genai.GenerativeModel('gemini-2.5-flash-lite')


# testing if it works
print(model.generate_content("Is sky blue?").text)

Yes, the sky is generally perceived as **blue**.

The reason for this is a phenomenon called **Rayleigh scattering**. When sunlight enters the Earth's atmosphere, it's made up of all the colors of the rainbow. Air molecules (like nitrogen and oxygen) are much smaller than the wavelengths of visible light. These molecules scatter shorter wavelengths of light (like blue and violet) more effectively than longer wavelengths (like red and orange).

Our eyes are more sensitive to blue light than violet, and also, some of the violet light is absorbed by the upper atmosphere. Therefore, we see the sky as blue.

However, the sky isn't *always* blue. It can appear:

*   **Red, orange, or pink** during sunrise and sunset, when the sunlight has to travel through more atmosphere, scattering away most of the blue light.
*   **Gray or white** on cloudy days, as water droplets and ice crystals in clouds scatter all colors of light roughly equally.
*   **Black** at night, when there's no direct sunligh

In [ ]:
# ran this to see available models -- ended up choosing the 2.5-flash-lite
for m in genai.list_models():
  if 'generateContent' in m.supported_generation_methods:
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gem

In [ ]:
import pandas as pd
import json
import time
from google.api_core import exceptions


df = pd.read_csv("marimba_mallets_dataset.csv")

SYSTEM_PROMPT = """You are an expert percussion educator. Convert the provided mallet specs into 3 realistic Q&A pairs.
1. 'Scenario': A student asking for a recommendation for a piece or venue.
2. 'Comparison': Comparing this mallet to another based on hardness or core.
3. 'Technical': Suitability for Stevens vs Burton grip or bar materials (Rosewood/Padauk).

Format the output ONLY as a JSON list of objects: [{"instruction": "...", "output": "..."}]
Do not include any introductory text or markdown formatting other than the JSON block."""

# looping but has retry logic in case
fine_tuning_data = []
output_file = "mallet_instruction_data.jsonl"

print(f"Starting query generation for {len(df)} mallets...")

for i, row in df.iterrows():
    # checks if name is already in fine_tuning_data
    # allows for rerunning of loop without redoing already finished mallets
    print(f"[{i+1}/{len(df)}] Processing: {row['Name']}")
    user_content = f"Mallet: {row['Name']}. Specs: {row['Description']}"

    success = False
    for attempt in range(3):
        try:
            response = model.generate_content(
                [SYSTEM_PROMPT, user_content],
                generation_config={"temperature": 0.7},
                request_options={"timeout": 30}
            )

            res_text = response.text
            if "```json" in res_text:
                res_text = res_text.split("```json")[1].split("```")[0]
            elif "```" in res_text:
                res_text = res_text.split("```")[1].split("```")[0]

            pairs = json.loads(res_text.strip())

            for pair in pairs:
                pair['input'] = f"Context: {row['Name']}"
                fine_tuning_data.append(pair)

            print(f"   ✅ Success: Generated {len(pairs)} pairs.")
            success = True
            break

        except Exception as e:
            if "429" in str(e):
                print(f"   ⏳ Quota hit. Sleeping for 60 seconds to reset...")
                time.sleep(60) # Waits a full minute if it hits the limit
            else:
                print(f"   ⚠️ Attempt {attempt+1} failed: {e}")
                time.sleep(5)

    if not success:
        print(f"   ❌ Skipping {row['Name']}")

    # Autosaves every 10 mallets
    if (i + 1) % 10 == 0:
        with open(output_file, "w") as f:
            for entry in fine_tuning_data:
                f.write(json.dumps(entry) + "\n")
        print(f"--- 💾 Progress saved to {output_file} ({len(fine_tuning_data)} total pairs) ---")

    # 4.5 seconds to not exceed 15 requests in 60 seconds to stay within free teir limits
    #time.sleep(4.5)

# Exporting
with open(output_file, "w") as f:
    for entry in fine_tuning_data:
        f.write(json.dumps(entry) + "\n")

print(f"\n FINISHED! Created {len(fine_tuning_data)} examples in {output_file}")

Starting query generation for 456 mallets...
[1/456] Processing: Innovative Soloist Series IP240 Medium Birch Marimba Mallets
   ✅ Success: Generated 3 pairs.
[2/456] Processing: Vic Firth Robert Van Sice Marimba Mallets - Medium
   ✅ Success: Generated 3 pairs.
[3/456] Processing: Vic Firth Robert Van Sice Keyboard Mallets - Medium
   ✅ Success: Generated 3 pairs.
[4/456] Processing: Innovative Ludwig Albert IP3106B Medium Hard Birch Marimba Mallets
   ✅ Success: Generated 3 pairs.
[5/456] Processing: Vic Firth Robert Van Sice Marimba Mallets - Medium Soft
   ✅ Success: Generated 3 pairs.
[6/456] Processing: Liberty One LMM Medium Yarn Birch Marimba Mallets
   ✅ Success: Generated 3 pairs.
[7/456] Processing: Marimba One Double Helix DHB3 Medium Hard Birch Mallets
   ✅ Success: Generated 3 pairs.
[8/456] Processing: Marimba One Double Helix DHB4 Medium Birch Mallets
   ✅ Success: Generated 3 pairs.
[9/456] Processing: Innovative Rattan Series RS251 Medium Rattan Vibe/Marimba Mallets
 

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2404.37ms


   ✅ Success: Generated 3 pairs.
[35/456] Processing: Malletech Late Nite Series Super Soft Mallets - Birch Handles
   ✅ Success: Generated 3 pairs.
[36/456] Processing: Encore NZ4B Nancy Zeltsman Medium Birch Marimba Mallets
   ✅ Success: Generated 3 pairs.
[37/456] Processing: Vic Firth Robert Van Sice Keyboard Mallets - Soft
   ✅ Success: Generated 3 pairs.
[38/456] Processing: Malletech Mtech Marimba Mallets - Medium
   ✅ Success: Generated 3 pairs.
[39/456] Processing: Innovative Rattan Series RS301 Hard Rattan Vibe/Marimba Mallets
   ✅ Success: Generated 3 pairs.
[40/456] Processing: Malletech Eric Sammut ES16 Medium to Medium Hard Birch Marimba Mallets
   ✅ Success: Generated 3 pairs.
--- 💾 Progress saved to mallet_instruction_data.jsonl (120 total pairs) ---
[41/456] Processing: Innovative Pius Cheung Graduated Marimba Mallet Set - Birch
   ✅ Success: Generated 3 pairs.
[42/456] Processing: Marimba One Wave Wrap WWXB3 Medium Birch Mallets
   ✅ Success: Generated 3 pairs.
[43/45

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 886.73ms


   ✅ Success: Generated 3 pairs.
[65/456] Processing: Malletech Chamber Series CH14 Medium Birch Marimba Mallets
   ✅ Success: Generated 3 pairs.
[66/456] Processing: Vic Firth American Custom Keyboard - Medium Hard Rubber
   ✅ Success: Generated 3 pairs.
[67/456] Processing: Marimba One Double Helix DHR3 Medium Hard Rattan Mallets
   ✅ Success: Generated 3 pairs.
[68/456] Processing: Innovative Ludwig Albert IP3103B Soft Birch Marimba Mallets
   ✅ Success: Generated 3 pairs.
[69/456] Processing: Innovative Casey Cangelosi CGL2 Rich Articulate Marimba Mallets
   ✅ Success: Generated 3 pairs.
[70/456] Processing: Innovative Thomas Burritt TB3 Medium Marimba Mallets
   ✅ Success: Generated 3 pairs.
--- 💾 Progress saved to mallet_instruction_data.jsonl (210 total pairs) ---
[71/456] Processing: Innovative Pius Cheung PIUS4B Medium Hard Marimba Mallets - Birch
   ✅ Success: Generated 3 pairs.
[72/456] Processing: Innovative William Moersch IP503 Med Hard Marimba Mallets
   ✅ Success: Gener

...

   ✅ Success: Generated 3 pairs.
[238/456] Processing: Innovative Weichen Lin Marimba Mallets Birch - Bass
   ✅ Success: Generated 3 pairs.
[239/456] Processing: Innovative Artisan Series - Hard Marimba Mallet - Cedar
   ✅ Success: Generated 3 pairs.
[240/456] Processing: Innovative Ivan Trevino Bass Marimba Mallets
   ✅ Success: Generated 3 pairs.
--- 💾 Progress saved to mallet_instruction_data.jsonl (720 total pairs) ---
[241/456] Processing: Innovative Eriko Daimo Series Medium Soft Marimba Mallets - Yarn Wrapped
   ✅ Success: Generated 3 pairs.
[242/456] Processing: Encore 32AYR Yarn Wound Series General Rattan Mallets
   ✅ Success: Generated 3 pairs.
[243/456] Processing: Malletech Leigh Howard Stevens LS15L Medium Soft-Hard Birch Marimba Mallets
   ✅ Success: Generated 3 pairs.
[244/456] Processing: Balter Basics Series Birch BB3 Soft Marimba Mallets Red Yarn
   ✅ Success: Generated 3 pairs.
[245/456] Processing: Schlagkraft Zivkovic Marimba Mallets - Blue Strong General (Cedar)


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 6776.21ms


   ✅ Success: Generated 3 pairs.
[327/456] Processing: Innovative Ludwig Albert IP3102B Med Extra Soft Birch Marimba Mallets
   ✅ Success: Generated 3 pairs.
[328/456] Processing: Marimba One Lynn Vartan LVB1 Hard Birch Marimba Mallets
   ✅ Success: Generated 3 pairs.
[329/456] Processing: Innovative Ludwig Albert IP3101B Extra Soft Birch Marimba Mallets
   ✅ Success: Generated 3 pairs.
[330/456] Processing: Encore NM4R Nanae Mimura Medium Rattan Marimba Mallets
   ✅ Success: Generated 3 pairs.
--- 💾 Progress saved to mallet_instruction_data.jsonl (990 total pairs) ---
[331/456] Processing: Salyers Jeff Moore JM104 Marimba Mallet - Hard
   ✅ Success: Generated 3 pairs.
[332/456] Processing: Promark SPYR Soft Marimba Mallets
   ✅ Success: Generated 3 pairs.
[333/456] Processing: Marimba One Round Sound Rattan Marimba Mallets - Medium Hard
   ✅ Success: Generated 3 pairs.
[334/456] Processing: Encore Kana Omori Signature Medium Marimba Mallets - Birch
   ✅ Success: Generated 3 pairs.
[33

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2681.45ms


   ✅ Success: Generated 3 pairs.
[377/456] Processing: Salyers Performance Collection Rattan Mallets - Hard Rubber
   ✅ Success: Generated 3 pairs.
[378/456] Processing: Encore NZ8B Nancy Zeltsman Boomer Bass Birch Marimba Mallets
   ✅ Success: Generated 3 pairs.
[379/456] Processing: Encore Kana Omori Signature Medium Hard Marimba Mallets - Birch
   ✅ Success: Generated 3 pairs.
[380/456] Processing: Encore 13ALR Latex Wrapped Medium Rattan Mallets
   ✅ Success: Generated 3 pairs.
--- 💾 Progress saved to mallet_instruction_data.jsonl (1140 total pairs) ---
[381/456] Processing: Yamaha Educational Series Medium Hard Marimba Mallets
   ✅ Success: Generated 3 pairs.
[382/456] Processing: Yamaha Keiko Abe Signature Birch Keyboard Mallets Hard
   ✅ Success: Generated 3 pairs.
[383/456] Processing: Schlagkraft Zivkovic Marimba Mallets - Lilac Tender General (Cedar)
   ✅ Success: Generated 3 pairs.
[384/456] Processing: Liberty One Scholastic Series Hard Marimba Mallets
   ✅ Success: Generat

## Method of Movement Queries
Marimba method and theory book

In [ ]:
import json
import time

pdf_path = "Method_of_Movement.pdf"
print(f"Uploading {pdf_path}...")

try:
    sample_file = genai.upload_file(path=pdf_path, mime_type="application/pdf")
    while sample_file.state.name == "PROCESSING":
        print(".", end="", flush=True)
        time.sleep(2)
        sample_file = genai.get_file(sample_file.name)
    print("\n✅ PDF Processed.")
except Exception as e:
    print(f"\n❌ Upload Error: {e}")
    raise

# Quering Loop
technique_data = []
topics = [
    "Grip mechanics, hand positioning, and the 'piston stroke'",
    "Single independent strokes and common accuracy errors",
    "Single alternating strokes and wrist rotation vs. arm movement",
    "Double vertical strokes and interval changes",
    "Double alternating strokes and the physics of the 'roll'"
]

for topic in topics:
    print(f" Extracting: {topic}...")

    prompt = f"""
    Using the uploaded 'Method of Movement' PDF, generate 15 unique Q&A pairs
    specifically about: {topic}.
    Focus on the nuances that a college-level percussionist would need to know.
    Format as a JSON list of objects: [{{ "instruction": "...", "output": "..." }}]
    """

    try:
        response = model.generate_content([prompt, sample_file])
        res_text = response.text

        # Clean markdown
        if "```json" in res_text:
            res_text = res_text.split("```json")[1].split("```")[0]
        elif "```" in res_text:
            res_text = res_text.split("```")[1].split("```")[0]

        # Parse and save
        batch = json.loads(res_text.strip())
        if isinstance(batch, list):
            technique_data.extend(batch)
        else:
            technique_data.append(batch)

        print(f"   ✅ Captured {len(batch) if isinstance(batch, list) else 1} pairs.")
        time.sleep(5)

    except Exception as e:
        print(f"   ❌ Skip {topic}: {e}")

# Export
output_file = "stevens_technique_expanded.jsonl"
if technique_data:
    with open(output_file, "w") as f:
        for entry in technique_data:
            f.write(json.dumps(entry) + "\n")
    print(f"\n SUCCESS! Saved {len(technique_data)} pairs to {output_file}")
else:
    print("\n❌ No data was collected.")

# Cleanup
genai.delete_file(sample_file.name)

Uploading Method_of_Movement.pdf...

✅ PDF Processed.
 Extracting: Grip mechanics, hand positioning, and the 'piston stroke'...
   ✅ Captured 15 pairs.
 Extracting: Single independent strokes and common accuracy errors...
   ✅ Captured 15 pairs.
 Extracting: Single alternating strokes and wrist rotation vs. arm movement...
   ✅ Captured 15 pairs.
 Extracting: Double vertical strokes and interval changes...
   ✅ Captured 15 pairs.
 Extracting: Double alternating strokes and the physics of the 'roll'...


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 6748.92ms


   ✅ Captured 15 pairs.

 SUCCESS! Saved 75 pairs to stevens_technique_expanded.jsonl


## Combining Data

Creating a 'master_training_data' file but upweighting the pedogagical data (stevens_technique files) so the model takes into account that data in an appropriate amount.

In [ ]:
import json
import random

# Load data
mallets = []
with open("mallet_instruction_data (1).jsonl", "r", encoding="utf-8-sig") as f:
    for line in f:
        if line.strip():
            mallets.append(json.loads(line))

stevens = []
for filename in ["stevens_technique_data.jsonl", "stevens_technique_expanded.jsonl"]:
    try:
        with open(filename, "r", encoding="utf-8-sig") as f:
            for i, line in enumerate(f, 1):
                stripped = line.strip()
                if not stripped:
                    continue
                try:
                    stevens.append(json.loads(stripped))
                except json.JSONDecodeError as e:
                    print(f"⚠️ Skipping bad line {i} in {filename}: {e}")
    except FileNotFoundError:
        print(f"⚠️ Warning: {filename} not found. Skipping.")

# Weighting logic
weighted_stevens = stevens * 5

# Combine and shuffle
final_data = mallets + weighted_stevens
random.shuffle(final_data)

# Save the final master file
with open("master_training_data.jsonl", "w", encoding="utf-8") as f:
    for entry in final_data:
        f.write(json.dumps(entry) + "\n")

print(f"Saved Dataset!")
print(f"Mallet Entries: {len(mallets)}")
print(f"Stevens Entries (Original): {len(stevens)}")
print(f"Stevens Entries (Weighted): {len(weighted_stevens)}")
print(f"Total entries in master_training_data.jsonl: {len(final_data)}")

⚠️ Skipping bad line 1 in stevens_technique_data.jsonl: Expecting value: line 1 column 1 (char 0)
Saved Dataset!
Mallet Entries: 1368
Stevens Entries (Original): 85
Stevens Entries (Weighted): 425
Total entries in master_training_data.jsonl: 1793


# Finetuning

In [ ]:
import sys
!{sys.executable} -m pip install -U transformers datasets accelerate trl peft torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 154.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 121.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 54.2 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.

In [ ]:
# formatting
from IPython.display import HTML, display

def set_css():
  display(HTML('''
  <style>
    pre {
        white-space: pre-wrap;
    }
  </style>
  '''))
get_ipython().events.register('pre_run_cell', set_css)

In [ ]:
import json
import random
from pathlib import Path
import requests

import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer # trainer library
from peft import LoraConfig

random.seed(42)
torch.manual_seed(42)

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print('Using device:', device)


Using device: cuda


Using unsloth hosted Llama 3.2 so we don't need to mess with hugging face access tokens and it's easy for collaboraters to run if needed.

In [ ]:
!pip install -U "bitsandbytes>=0.46.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 40.0 MB/s eta 0:00:00


In [ ]:
from transformers import BitsAndBytesConfig

checkpoint = "unsloth/Llama-3.2-3B-Instruct"

# 4-bit quantization config so 3B model fits on T4 GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Downloading and loading model into GPU memory (this takes ~1-2 mins)...")
model = AutoModelForCausalLM.from_pretrained(
    checkpoint,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

print('Loaded:', checkpoint)
print('Vocab size:', len(tokenizer))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

Loaded: unsloth/Llama-3.2-3B-Instruct
Vocab size: 128256


In [ ]:
# load master data
qa_examples = []
with open("master_training_data.jsonl", "r", encoding="utf-8-sig") as f:
    for line in f:
        if line.strip():
            qa_examples.append(json.loads(line))

# Format as a chat
def format_as_chat_text(example):
    # combines instruction and the input
    user_text = example['instruction']
    if 'input' in example and example['input'].strip():
        user_text += f"\n\nContext: {example['input']}"

    messages = [
        {
            'role': 'system',
            'content': 'You are a master percussion educator and equipment expert. Answer with technical accuracy and musical insight.'
        },
        {
            'role': 'user',
            'content': user_text
        },
        {
            'role': 'assistant',
            'content': example['output']
        }
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

formatted_examples = [{'text': format_as_chat_text(ex), **ex} for ex in qa_examples]

# creates 90/10 train/eval split
split = int(0.90 * len(formatted_examples))
train_rows = formatted_examples[:split]
eval_rows = formatted_examples[split:]

train_dataset = Dataset.from_list(train_rows)
eval_dataset = Dataset.from_list(eval_rows)

print("Sample formatted text:")
print(train_dataset[0]['text'])
print('\ntrain examples:', len(train_dataset))
print('eval examples:', len(eval_dataset))

Sample formatted text:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 06 May 2026

You are a master percussion educator and equipment expert. Answer with technical accuracy and musical insight.<|eot_id|><|start_header_id|>user<|end_header_id|>

When playing single independent strokes, what is the recommended relationship between the mallet head and the keyboard in terms of height, and why is this important?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

The mallet head should be held relatively close to the keyboard, about 1/2 inch above it. This proximity is crucial for minimizing extraneous motion and maximizing efficiency, particularly in faster passages.<|eot_id|>

train examples: 1613
eval examples: 180


In [ ]:
# LoRA Configuration
peft_config = LoraConfig(
    r=16, # slightly higher rank for the complex pedagogy concepts
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)

# Training Configuration
training_args = SFTConfig(
    output_dir='marimba_expert_lora',
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy='no',
    eval_strategy='no', # set to 'steps' if we want to test against eval_dataset during the training
    max_length=1024,
    report_to='none',fp16=False,
    bf16=True,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
)

Adding EOS to train dataset:   0%|          | 0/1613 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1613 [00:00<?, ? examples/s]

In [ ]:
# Run fine tuning
print("Starting fine-tuning process...")
trainer.train()
print("Training complete!")

Starting fine-tuning process...


Step,Training Loss
10,2.557788
20,1.665308
30,1.451541
40,1.365126
50,1.289649
60,1.269108
70,1.237688
80,1.207357
90,1.213726
100,1.199712


Training complete!


# Exporting Model

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch, gc
from transformers import AutoModelForCausalLM
from peft import PeftModel

# Save LoRA adapters locally
print("1. Saving the new percussion knowledge...")
trainer.model.save_pretrained("lora_adapter")
tokenizer.save_pretrained("lora_adapter")

# Clear out the training model from GPU memory
print("2. Clearing GPU memory...")
del model
del trainer
gc.collect()
torch.cuda.empty_cache()

# Reload base model natively without compression
print("3. Reloading base model natively...")
base_model = AutoModelForCausalLM.from_pretrained(
    "unsloth/Llama-3.2-3B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto"
)

# Merge adapter permanently into the base model
print("4. Stitching knowledge into base model...")
merged_model = PeftModel.from_pretrained(base_model, "lora_adapter")
merged_model = merged_model.merge_and_unload()

# Save the final standard Hugging Face model
print("5. Saving merged model...")
merged_model.save_pretrained("merged_model")
tokenizer.save_pretrained("merged_model")
print("✅ Model merged and ready for GGUF compilation!")

1. Saving the new percussion knowledge...
2. Clearing GPU memory...
3. Reloading base model natively...


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

4. Stitching knowledge into base model...
5. Saving merged model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model merged and ready for GGUF compilation!


Ran into a lot of difficulties trying to export the model, I used Gemini a lot to help work through my issues. Nearly all of the code below for the rest of this section is completely AI generated to help get the model exported to my Google Drive.

In [ ]:
# # Download the official Llama.cpp tools and requirements
# !git clone https://github.com/ggerganov/llama.cpp
# !pip install -r llama.cpp/requirements.txt

# Run the compiler and send it directly to your mounted Google Drive
!python llama.cpp/convert_hf_to_gguf.py merged_model \
  --outtype q8_0 \
  --outfile /content/drive/MyDrive/marimba_expert_v1.gguf

!echo "✅ Model successfully compiled and saved to Google Drive!"

INFO:hf-to-gguf:Loading model: merged_model
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:rope_freqs.weight,           torch.float32 --> F32, shape = {64}
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> Q8_0, shape = {3072, 128256}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float16 --> F32, shape = {3072}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float16 --> Q8_0, shape = {8192, 3072}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float16 --> Q8_0, shape = {3072, 8192}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float16 --> Q8_0, shape = {3072, 8192}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.float16 --> F32, shape = {3072}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.float16 --> Q8_0, shape = {3072, 1024}
INFO:hf-to-gguf:blk.0.attn_output.weig

In [ ]:
import shutil

print("Backing up the fine-tuned model to Google Drive...")
# Copies the entire merged_model folder to your Drive
shutil.copytree("merged_model", "/content/drive/MyDrive/marimba_expert_raw_backup", dirs_exist_ok=True)

print("✅ Model is officially safe in your Google Drive!")

Backing up the fine-tuned model to Google Drive...
✅ Model is officially safe in your Google Drive!


In [ ]:
# 1. Nuke the broken vision library so it stops interrupting
!pip uninstall -y torchvision

# 2. Install the specific vocabulary compiler for Llama 3
!pip install tiktoken

# 3. Run the GGUF compiler again
!python llama.cpp/convert_hf_to_gguf.py merged_model \
  --outtype q8_0 \
  --outfile /content/drive/MyDrive/marimba_expert_v1.gguf && echo "✅ NOW it is successfully compiled and saved to Google Drive!"

Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
INFO:hf-to-gguf:Loading model: merged_model
INFO:numexpr.utils:NumExpr defaulting to 12 threads.
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:rope_freqs.weight,           torch.float32 --> F32, shape = {64}
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> Q8_0, shape = {3072, 128256}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float16 --> F32, shape = {3072}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float16 --> Q8_0, shape = {8192, 3072}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float16 --> Q8_0, shape = {3072, 8192}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float16 --> Q8_0, shape = {3072, 8192}
INFO:hf-to-gg

# Testing exported model


In [ ]:
!CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python

  Using cached llama_cpp_python-0.3.22.tar.gz (68.0 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached diskcache-5.6.3-py3-none-any.whl.metadata (20 kB)
Using cached diskcache-5.6.3-py3-none-any.whl (45 kB)
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.22-py3-none-linux_x86_64.whl size=135859096 sha256=458380c1bd61d0a7cd69128c8580b2a617c8776a0a848a95e88104338aba9904
  Stored in directory: /root/.cache/pip/wheels/c2/31/5c/91da2c279c89f857a64ce732ef0d26f5888a38ac9022526607
Successfully built llama-cpp-python


In [ ]:
from llama_cpp import Llama

# Load GGUF model from Google Drive
print("Loading Marimba Expert GGUF into memory...")
llm = Llama(
    model_path="/content/drive/MyDrive/Marimba Expert Model/marimba_expert_v1.gguf",
    n_gpu_layers=-1, # Offload calculation layers to the GPU
    n_ctx=2048,      # Context window size
    verbose=False    # Hides C++ compiler logs
)

# Define the Chat Function
def ask_expert(question):
    print(f" Question: {question}")

    # use create_chat_completion so to format text using Llama 3 Chat template
    response = llm.create_chat_completion(
        messages=[
            {
                "role": "system",
                "content": "You are a master percussion educator and equipment expert. Answer with technical accuracy and musical insight."
            },
            {
                "role": "user",
                "content": question
            }
        ],
        max_tokens=512,
        temperature=0.3, # Keep this low (0.1 to 0.3) so it relies on facts
    )

    answer = response['choices'][0]['message']['content']
    print(f"\n Chatbot: {answer}")
    print("-" * 60)

print("\n Starting tests...\n")

# asking about technique
ask_expert("Explain the concept of the 'piston stroke' and why it is more efficient than an elliptical motion.")

# asking about a product
ask_expert("I'm playing a lyrical solo on a rosewood marimba and need something for the middle register. What IP mallet do you recommend?")

# asking about grip type
ask_expert("If I am using Stevens grip, how do my fingers control the interval changes?")

Loading Marimba Expert GGUF into memory...


llama_context: n_ctx_seq (2048) < n_ctx_train (131072) -- the full capacity of the model will not be utilized



 Starting tests...

 Question: Explain the concept of the 'piston stroke' and why it is more efficient than an elliptical motion.

 Chatbot: The 'piston stroke' is a fundamental concept in single alternating stroke technique, emphasizing a straight vertical motion of the mallet head. This is more efficient than an elliptical motion because it allows for a more direct and powerful impact on the bar, reducing unnecessary energy expenditure. The piston stroke's efficiency comes from its ability to maintain a consistent angle of incidence and minimize the time spent in the air, resulting in a more efficient transfer of energy to the bar.
------------------------------------------------------------
 Question: I'm playing a lyrical solo on a rosewood marimba and need something for the middle register. What IP mallet do you recommend?

 Chatbot: For a rosewood marimba and a lyrical solo in the middle register, I'd highly recommend the Innovative Percussion IP310. Its medium-hardness and bala